# Association Rule Learning

Association rule learning is **unsupervised pattern mining**: given a set of transactions (baskets), find rules of the form *"people who buy X tend to also buy Y"* — without any labels to learn from. It powers supermarket shelf planning, cross-sell recommendations, "frequently bought together" widgets, and click-stream analysis.

> **Prerequisites:** none — this topic is self-contained and needs only Level 1 Python + pandas. It is the classic entry point into unsupervised pattern mining, alongside clustering (Level 2, Section 6).


## 1. The Problem: Finding Co-Purchase Patterns

The input is a **transaction database**: each row is one basket, each basket is a set of items. The output is a set of **rules** $X \Rightarrow Y$ (read: "$X$ implies $Y$") scored by three metrics. A famous (real) example from retail: *diapers ⇒ beer* — the story may be urban legend, but the analysis pattern it represents is used every day in e-commerce.

| Setting | Example question |
|---|---|
| Retail | Which items should the "buy again / pairs well with" widget show? |
| Streaming / media | Which content is consumed together in one session? |
| Web analytics | Which pages are visited in the same session → site re-design? |
| Medicine | Which symptoms co-occur → symptom checkers, alerting rules |


## 2. The Three Metrics

Every rule $X \Rightarrow Y$ is scored by **support**, **confidence**, and **lift**:

$$
\text{support}(X) = \frac{\#\ \text{baskets containing } X}{\text{total baskets}}, \qquad
\text{confidence}(X \Rightarrow Y) = \frac{\text{support}(X \cup Y)}{\text{support}(X)}, \qquad
\text{lift}(X \Rightarrow Y) = \frac{\text{confidence}(X \Rightarrow Y)}{\text{support}(Y)}
$$

| Metric | Answers | Range |
|---|---|---|
| **Support** | How common is the itemset at all? | $[0, 1]$ |
| **Confidence** | Given $X$ was bought, how often is $Y$ bought too? | $[0, 1]$ — a conditional probability $P(Y \mid X)$ |
| **Lift** | How much more likely is $Y$ given $X$, compared to its base rate? | $\text{lift} = 1$ means independent; $>1$ positive association; $<1$ negative |

> **Worked micro-example.** 100 baskets; 10 contain both *bread* and *butter*; 20 contain bread... wait — 20 contain *butter* overall, 25 contain *bread* overall. Then support(bread ⇒ butter) = 10/100 = 0.10, confidence = 10/25 = 0.40, and lift = 0.40 / 0.20 = **2.0** — buying bread doubles the odds of butter. If lift came out ≈ 1, the pair is co-occurring only by chance.


## 3. The Dataset

We use the classic **Market Basket Optimisation** dataset: **7,501 supermarket transactions** (one week, a French supermarket) with **119 unique items**, each basket holding up to 20 items. The file ships next to this notebook (`Market_Basket_Optimisation.csv`) so everything below is fully reproducible offline.

First step: parse the CSV into a list of baskets (rows have a variable number of non-empty slots), then one-hot encode into the transactions matrix the algorithms consume.


In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

df = pd.read_csv("Market_Basket_Optimisation.csv", header=None)
print(f"{df.shape[0]:,} transactions, up to {df.shape[1]} items per basket")

transactions = [[item for item in row if isinstance(item, str)] for row in df.values.tolist()]
print(f"Example basket: {transactions[0][:5]} ... ({len(transactions[0])} items)")

te = TransactionEncoder()
one_hot = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)
print(f"One-hot matrix: {one_hot.shape[0]:,} x {one_hot.shape[1]}")

one_hot.mean().sort_values(ascending=False).head(8).round(3)

7,501 transactions, up to 20 items per basket
Example basket: ['shrimp', 'almonds', 'avocado', 'vegetables mix', 'green grapes'] ... (20 items)
One-hot matrix: 7,501 x 120


mineral water    0.238
eggs             0.180
spaghetti        0.174
french fries     0.171
chocolate        0.164
green tea        0.132
milk             0.130
ground beef      0.098
dtype: float64

## 4. The Apriori Algorithm

Brute-forcing all itemsets is $2^{119}$ combinations — impossible. **Apriori** (Agrawal & Srikant, 1994) prunes using the **Apriori property** (anti-monotonicity):

> *Every subset of a frequent itemset must itself be frequent.* Equivalently: if {milk, eggs} is frequent, then {milk} and {eggs} are each frequent; if {milk} is *rare*, no superset containing milk can be frequent.

So Apriori searches **level-wise**: find frequent 1-itemsets → use only those to build candidate 2-itemsets → count → build candidates for level 3 from frequent level-2 pairs → stop when a level comes back empty. The candidate space collapses from exponential to (in practice) tiny.


In [2]:
# Support here means: itemset appears in at least 2% of the 7,501 baskets (>= 150 baskets)
frequent = apriori(one_hot, min_support=0.02, use_colnames=True)
print(f"{len(frequent)} frequent itemsets at min_support = 0.02\n")

frequent["n_items"] = frequent["itemsets"].apply(len)
frequent.sort_values("support", ascending=False).head(8).round(3)

103 frequent itemsets at min_support = 0.02



,support,itemsets,n_items
34,0.238,frozenset({mineral water}),1
13,0.180,frozenset({eggs}),1
44,0.174,frozenset({spaghetti}),1
17,0.171,frozenset({french fries}),1
9,0.164,frozenset({chocolate}),1
24,0.132,frozenset({green tea}),1
33,0.130,frozenset({milk}),1
25,0.098,frozenset({ground beef}),1


## 5. From Frequent Itemsets to Rules

An itemset {A, B} yields two rules: $A \Rightarrow B$ and $B \Rightarrow A$. `association_rules` enumerates them and scores each. The standard workflow filters on **lift > 1** (positive association), then sorts by confidence or lift depending on whether you care about *reliability* or *strength*.


In [3]:
rules = association_rules(frequent, metric="lift", min_threshold=1.5)
rules = rules.sort_values("lift", ascending=False)
print(f"{len(rules)} rules with lift >= 1.5\n")

cols = ["antecedents", "consequents", "antecedent support", "consequent support",
        "support", "confidence", "lift"]
rules[cols].head(10).round(3)

36 rules with lift >= 1.5



,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
19,frozenset({spaghetti}),frozenset({ground beef}),0.174,0.098,0.039,0.225,2.291
18,frozenset({ground beef}),frozenset({spaghetti}),0.098,0.174,0.039,0.399,2.291
28,frozenset({olive oil}),frozenset({spaghetti}),0.066,0.174,0.023,0.348,2.000
29,frozenset({spaghetti}),frozenset({olive oil}),0.174,0.066,0.023,0.132,2.000
27,frozenset({mineral water}),frozenset({soup}),0.238,0.051,0.023,0.097,1.915
26,frozenset({soup}),frozenset({mineral water}),0.051,0.238,0.023,0.456,1.915
9,frozenset({milk}),frozenset({frozen vegetables}),0.130,0.095,0.024,0.182,1.910
8,frozenset({frozen vegetables}),frozenset({milk}),0.095,0.130,0.024,0.248,1.910
1,frozenset({eggs}),frozenset({burgers}),0.180,0.087,0.029,0.160,1.838
0,frozenset({burgers}),frozenset({eggs}),0.087,0.180,0.029,0.330,1.838


## 6. Reading Rules Like an Analyst

The strongest rule here is **spaghetti ⇒ ground beef** (lift ≈ 2.29): ground beef appears in ~9.8% of baskets overall, but in ~22.5% of spaghetti baskets. A merchandiser would pair them (meal-deal, adjacent shelf); a recommender would rank ground beef high on the spaghetti product page.

Two rules you must not skip:

1. **Confidence is asymmetric, lift is symmetric.** P(ground beef | spaghetti) = 0.225 but P(spaghetti | ground beef) = 0.399 — both directions share one lift value. Decide which direction matters for your use case (recommend Y *to* buyers of X) and read that row's confidence.
2. **High confidence can be trivial.** ⇒ eggs has confidence 0.33 because eggs are everywhere (support 0.18). A rule towards a near-universal consequent is business-noise; lift (and judgment) filters it.


In [4]:
# Same pair, both directions — confidence flips, lift does not
pair = rules[rules["antecedents"].apply(lambda s: "spaghetti" in s) &
             rules["consequents"].apply(lambda s: "ground beef" in s)]
pair = pd.concat([pair, rules[rules["antecedents"].apply(lambda s: "ground beef" in s) &
                              rules["consequents"].apply(lambda s: "spaghetti" in s)]])
pair[cols].round(3)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
19,frozenset({spaghetti}),frozenset({ground beef}),0.174,0.098,0.039,0.225,2.291
18,frozenset({ground beef}),frozenset({spaghetti}),0.098,0.174,0.039,0.399,2.291


## 7. Eclat and FP-Growth

**Eclat** (*Équivalence CLAss Transformation*, Zaki et al., 1997) reformulates mining in the **vertical format**: instead of "basket → items", it stores "item → set of transaction IDs" (TID-lists). Support of an itemset = size of the *intersection* of TID-lists — pure set intersection, no database scans. Eclat produces frequent itemsets (not rules directly) and shines when itemsets are the goal and TID-lists stay small. It is the textbook alternative to Apriori; `mlxtend` implements its successor instead.

**FP-Growth** (Han, Pei & Yin, 2000) is the practical successor of both. It compresses the whole database into a **frequent-pattern (FP) tree** — a prefix tree that stores each item's occurrence count — and mines frequent itemsets from the tree directly, **without candidate generation**. In `mlxtend` you swap one function call and get the identical result set:

> The theoretical selling point of FP-Growth is two database passes (vs. one per level for Apriori) and no candidate generation — the advantage grows with database size and density. On a small, sparse table like this one, mlxtend's highly-optimised Apriori can even be faster; the honest takeaway is *both are milliseconds here, profile on your own data at scale*. For truly large data, the FP-Growth implementation in **MLlib (PySpark)** is the standard production route.


In [5]:
import time

t0 = time.perf_counter()
ap_itemsets = apriori(one_hot, min_support=0.02)
t_apriori = time.perf_counter() - t0

t0 = time.perf_counter()
fp_itemsets = fpgrowth(one_hot, min_support=0.02)
t_fpgrowth = time.perf_counter() - t0

print(f"Apriori:   {t_apriori*1000:6.1f} ms -> {len(ap_itemsets)} itemsets")
print(f"FP-Growth: {t_fpgrowth*1000:6.1f} ms -> {len(fp_itemsets)} itemsets")
print(f"Same result set: {set(map(frozenset, ap_itemsets['itemsets'])) == set(map(frozenset, fp_itemsets['itemsets']))}")

Apriori:     16.5 ms -> 103 itemsets
FP-Growth:   47.8 ms -> 103 itemsets
Same result set: True


## 8. Beyond the Supermarket

- **Recommendation systems.** Association rules are the simplest recommender: rank consequents of the user's basket. The roadmap's next step up is collaborative filtering — see the [Movie Recommender System](https://github.com/AdilShamim8/Movie-Recommender-System) and [Book Recommender System](https://github.com/AdilShamim8/Book-Recommender-System) projects in Level 4.
- **Scale.** Millions of baskets → PySpark MLlib `FPGrowth`, which distributes tree mining across a cluster and is the industry standard.
- **Session data.** Same algorithms apply to click-streams and playlists — a "basket" is just a set of items grouped by a session or user.
- **Caution.** Rules describe **correlation, not causation** — a lift of 2.29 does not mean spaghetti *causes* ground beef purchases. Use them for ranking and merchandising, not causal claims.


## Key Takeaways

- **Association rule mining is unsupervised** — no labels, just co-occurrence structure across sets.
- The three metrics: **support** (how common), **confidence** ($P(Y \mid X)$, directional), **lift** ($\frac{P(Y\mid X)}{P(Y)}$, symmetric, 1 = independence).
- **Apriori** prunes the search with the anti-monotone Apriori property; **Eclat** mines via vertical TID-list intersections; **FP-Growth** compresses the database into a prefix tree and skips candidate generation entirely.
- Read rules in the **direction your product acts on**, and treat high-confidence-toward-ubiquitous-items rules as noise.
- In `mlxtend`: `TransactionEncoder` → `apriori`/`fpgrowth` → `association_rules(metric="lift")` is the whole pipeline.
- Next in this roadmap: **Anomaly Detection** (the other classic unsupervised task) and the Level 5 end-to-end project.
